In [ ]:
!pip install __future__

In [1]:
# app.py
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Dict, Optional

import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
from fredapi import Fred


ModuleNotFoundError: No module named 'streamlit'

In [ ]:
# ----------------------------
# Page config
# ----------------------------
st.set_page_config(
    page_title="UK Economic Pulse",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.title("UK Economic Pulse")
st.caption("A live macro dashboard answering one question: Is the UK economy strengthening or weakening right now?")

In [ ]:
# ----------------------------
# FRED setup
# ----------------------------
FRED_API_KEY = st.secrets.get("FRED_API_KEY", os.getenv("FRED_API_KEY"))

if not FRED_API_KEY:
    st.error("Missing FRED_API_KEY. Add it to Streamlit secrets or your environment.")
    st.stop()

fred = Fred(api_key=FRED_API_KEY)

In [ ]:
# ----------------------------
# Config
# ----------------------------
@dataclass
class SeriesSpec:
    code: str
    category: str
    label: str
    transform: str  # 'level', 'yoy_pct', 'mom_pct', 'diff', 'spread'
    higher_is_better: bool
    weight: float
    enabled: bool = True


SERIES_CONFIG: Dict[str, SeriesSpec] = {
    # Economic Activity
    "gdp_growth": SeriesSpec(
        code="NAEXKP01GBQ189S",
        category="Economic Activity",
        label="GDP Growth Rate",
        transform="level",
        higher_is_better=True,
        weight=1.5,
    ),
    "retail_sales": SeriesSpec( # US
        code="RSXFSN",
        category="Economic Activity",
        label="Retail Sales",
        transform="yoy_pct",
        higher_is_better=True,
        weight=1.2,
    ),
    "house_prices": SeriesSpec( # UK
        code="QGBR628BIS",
        category="Economic Activity",
        label="House Prices",
        transform="yoy_pct",
        higher_is_better=True,
        weight=0.8,
    ),

    # Inflation Pressure
    "cpi": SeriesSpec(          # out of date, use ONS extract
        code="GBRCPIALLMINMEI",
        category="Inflation Pressure",
        label="CPI",
        transform="yoy_pct",
        higher_is_better=False,
        weight=1.3,
    ),
    "ppi": SeriesSpec(         # out of date, use ONS extract
        code="GBRPPIENGMINMEI",
        category="Inflation Pressure",
        label="Producer Price Index",
        transform="yoy_pct",
        higher_is_better=False,
        weight=0.8,
    ),
    "wage_growth": SeriesSpec( # out of date, use ONS extract 
        code="AVANOMGBQ657S",
        category="Inflation Pressure",
        label="Wage Growth",
        transform="yoy_pct",
        higher_is_better=False,
        weight=0.9,
    ),

    # Financial Conditions
    "policy_rate": SeriesSpec( # out of date, use 
        code="IRSTCB01GBM156N",
        category="Financial Conditions",
        label="Policy Rate",
        transform="level",
        higher_is_better=False,
        weight=1.2,
    ),
    "uk_10y_yield": SeriesSpec(
        code="IRLTLT01GBM156N",
        category="Financial Conditions",
        label="10Y Government Yield",
        transform="level",
        higher_is_better=False,
        weight=0.6,
    ),
    "oil": SeriesSpec(
        code="DCOILBRENTEU",
        category="Financial Conditions",
        label="Brent Oil",
        transform="yoy_pct",
        higher_is_better=False,
        weight=0.6,
    ),

    # Optional extras you can switch on later
    "unemployment": SeriesSpec(
        code="LRUNTTTTGBM156S",
        category="Economic Activity",
        label="Unemployment Rate",
        transform="level",
        higher_is_better=False,
        weight=1.3,
        enabled=False,
    ),
    "consumer_confidence": SeriesSpec(
        code="CSCICP03GBM665S",
        category="Economic Activity",
        label="Consumer Confidence",
        transform="level",
        higher_is_better=True,
        weight=0.8,
        enabled=False,
    ),
    "labour_participation": SeriesSpec(
        code="LFAC64TTGBM647S",
        category="Economic Activity",
        label="Labour Participation",
        transform="level",
        higher_is_better=True,
        weight=0.6,
        enabled=False,
    ),
    "gold": SeriesSpec(
        code="GOLDAMGBD228NLBM",
        category="Financial Conditions",
        label="Gold",
        transform="yoy_pct",
        higher_is_better=False,
        weight=0.2,
        enabled=False,
    ),
}

In [ ]:
# ----------------------------
# Sidebar controls
# ----------------------------
st.sidebar.header("Controls")

start_date = st.sidebar.date_input("Start date", value=pd.Timestamp("2014-01-01"))
end_date = st.sidebar.date_input("End date", value=pd.Timestamp.today().date())

enabled_keys = []
for key, spec in SERIES_CONFIG.items():
    checked = st.sidebar.checkbox(
        f"{spec.label} ({spec.code})",
        value=spec.enabled,
        key=f"chk_{key}",
    )
    if checked:
        enabled_keys.append(key)

if len(enabled_keys) < 3:
    st.warning("Enable at least 3 indicators so the composite signal has some spine.")
    st.stop()

In [ ]:
# ----------------------------
# Data utilities
# ----------------------------
@st.cache_data(show_spinner=False)
def fetch_series(code: str, start: str, end: str) -> pd.Series:
    s = fred.get_series(code, observation_start=start, observation_end=end)
    s = pd.Series(s).sort_index()
    s.index = pd.to_datetime(s.index)
    s = s.dropna()
    s.name = code
    return s


def to_monthly(s: pd.Series) -> pd.Series:
    """
    Bring everything onto a monthly backbone.
    Daily -> month end mean
    Quarterly -> month end forward-fill
    Monthly -> month end as-is
    """
    if s.empty:
        return s

    inferred = pd.infer_freq(s.index[: min(len(s.index), 10)])
    # fallback heuristic
    if inferred is None:
        median_days = s.index.to_series().diff().dt.days.median()
        if pd.isna(median_days):
            median_days = 30
        if median_days <= 7:
            freq_type = "dailyish"
        elif median_days <= 40:
            freq_type = "monthlyish"
        else:
            freq_type = "quarterlyish"
    else:
        inferred = inferred.upper()
        if inferred.startswith("D") or inferred.startswith("B") or inferred.startswith("W"):
            freq_type = "dailyish"
        elif inferred.startswith("M"):
            freq_type = "monthlyish"
        elif inferred.startswith("Q"):
            freq_type = "quarterlyish"
        else:
            freq_type = "monthlyish"

    if freq_type == "dailyish":
        out = s.resample("M").mean()
    elif freq_type == "quarterlyish":
        out = s.resample("M").ffill()
    else:
        out = s.resample("M").last()

    return out.dropna()


def transform_series(s: pd.Series, transform: str) -> pd.Series:
    if transform == "level":
        return s
    if transform == "yoy_pct":
        return s.pct_change(12) * 100
    if transform == "mom_pct":
        return s.pct_change(1) * 100
    if transform == "diff":
        return s.diff()
    raise ValueError(f"Unsupported transform: {transform}")


def robust_zscore(s: pd.Series, window: int = 60) -> pd.Series:
    """
    Rolling robust z-score using median and MAD.
    Keeps the model from becoming a hostage to one ugly regime shift.
    """
    roll_median = s.rolling(window).median()
    mad = (s - roll_median).abs().rolling(window).median()
    z = 0.6745 * (s - roll_median) / mad.replace(0, np.nan)
    return z.clip(-3, 3)


def indicator_score(signal: pd.Series, higher_is_better: bool) -> pd.Series:
    z = robust_zscore(signal)
    if not higher_is_better:
        z = -z
    return z


def verdict_from_score(x: float) -> str:
    if pd.isna(x):
        return "Insufficient Data"
    if x >= 0.5:
        return "Expansion"
    if x <= -0.5:
        return "Contraction"
    return "Neutral"


def verdict_color(verdict: str) -> str:
    if verdict == "Expansion":
        return "🟢"
    if verdict == "Contraction":
        return "🔴"
    if verdict == "Neutral":
        return "🟡"
    return "⚪"


def format_latest(v: Optional[float]) -> str:
    if v is None or pd.isna(v):
        return "n/a"
    return f"{v:,.2f}"

In [ ]:
# ----------------------------
# Load data
# ----------------------------
with st.spinner("Pulling macro data from FRED..."):
    raw_series = {}
    monthly_series = {}
    signal_series = {}
    score_series = {}
    meta_rows = []

    for key in enabled_keys:
        spec = SERIES_CONFIG[key]
        try:
            raw = fetch_series(spec.code, str(start_date), str(end_date))
            monthly = to_monthly(raw)
            signal = transform_series(monthly, spec.transform)
            score = indicator_score(signal, spec.higher_is_better)

            raw_series[key] = raw
            monthly_series[key] = monthly
            signal_series[key] = signal
            score_series[key] = score

            meta_rows.append(
                {
                    "key": key,
                    "label": spec.label,
                    "category": spec.category,
                    "code": spec.code,
                    "transform": spec.transform,
                    "higher_is_better": spec.higher_is_better,
                    "weight": spec.weight,
                    "latest_level": monthly.dropna().iloc[-1] if not monthly.dropna().empty else np.nan,
                    "latest_signal": signal.dropna().iloc[-1] if not signal.dropna().empty else np.nan,
                    "latest_score": score.dropna().iloc[-1] if not score.dropna().empty else np.nan,
                }
            )
        except Exception as e:
            st.warning(f"Failed to load {spec.label} ({spec.code}): {e}")

if not meta_rows:
    st.error("Nothing loaded. Either the FRED codes are wrong, the API key is broken, or the series source moved.")
    st.stop()

meta = pd.DataFrame(meta_rows).set_index("key")

# align and aggregate
all_scores = pd.concat(
    [score_series[k].rename(k) for k in score_series],
    axis=1
).sort_index()

weights = pd.Series({k: SERIES_CONFIG[k].weight for k in all_scores.columns})

weighted_scores = all_scores.mul(weights, axis=1)
composite = weighted_scores.sum(axis=1) / weights[all_scores.notna().any(axis=0)].sum()
# better monthly weighted mean allowing missing values row-by-row
valid_weight_matrix = (~all_scores.isna()).mul(weights, axis=1)
composite = weighted_scores.sum(axis=1) / valid_weight_matrix.sum(axis=1)

# category composites
category_map = {k: SERIES_CONFIG[k].category for k in score_series.keys()}
category_scores = {}
for category in sorted(set(category_map.values())):
    keys = [k for k, v in category_map.items() if v == category]
    sub = all_scores[keys]
    sub_weights = pd.Series({k: SERIES_CONFIG[k].weight for k in keys})
    sub_weighted = sub.mul(sub_weights, axis=1)
    sub_valid_weight_matrix = (~sub.isna()).mul(sub_weights, axis=1)
    category_scores[category] = sub_weighted.sum(axis=1) / sub_valid_weight_matrix.sum(axis=1)

latest_composite = composite.dropna().iloc[-1] if not composite.dropna().empty else np.nan
verdict = verdict_from_score(latest_composite)
badge = verdict_color(verdict)


In [ ]:



# ----------------------------
# Narrative
# ----------------------------
def build_narrative(meta_df: pd.DataFrame) -> str:
    pieces = []

    # Economic activity
    activity = meta_df[meta_df["category"] == "Economic Activity"]["latest_score"].mean()
    if pd.notna(activity):
        if activity > 0.4:
            pieces.append("Activity is still holding up")
        elif activity < -0.4:
            pieces.append("activity is rolling over")
        else:
            pieces.append("activity is mixed")

    # Inflation
    inflation = meta_df[meta_df["category"] == "Inflation Pressure"]["latest_score"].mean()
    if pd.notna(inflation):
        if inflation > 0.4:
            pieces.append("inflation pressure looks manageable")
        elif inflation < -0.4:
            pieces.append("inflation pressure remains sticky")
        else:
            pieces.append("inflation is easing, but not cleanly")

    # Financial conditions
    fin = meta_df[meta_df["category"] == "Financial Conditions"]["latest_score"].mean()
    if pd.notna(fin):
        if fin > 0.4:
            pieces.append("financial conditions are supportive")
        elif fin < -0.4:
            pieces.append("financial conditions are restrictive")
        else:
            pieces.append("financial conditions are neither loose nor crushing")

    sentence = ". ".join(pieces)
    if sentence:
        sentence += "."
    return sentence.capitalize()


narrative = build_narrative(meta)


# ----------------------------
# Top row
# ----------------------------
c1, c2, c3, c4 = st.columns([1.2, 1, 1, 1])

with c1:
    st.subheader("Verdict")
    st.markdown(f"## {badge} {verdict}")

with c2:
    st.metric("Composite Score", format_latest(latest_composite))

with c3:
    latest_date = composite.dropna().index.max() if not composite.dropna().empty else None
    st.metric("Latest Observation", latest_date.strftime("%Y-%m-%d") if latest_date else "n/a")

with c4:
    active_count = len(meta)
    st.metric("Indicators Active", str(active_count))

st.write(narrative)


# ----------------------------
# Composite chart
# ----------------------------
st.subheader("Composite Macro Signal")

fig = plt.figure(figsize=(12, 4.5))
ax = fig.add_subplot(111)
ax.plot(composite.index, composite.values)
ax.axhline(0.5, linestyle="--")
ax.axhline(0.0, linestyle=":")
ax.axhline(-0.5, linestyle="--")
ax.set_ylabel("Weighted Score")
ax.set_xlabel("Date")
ax.set_title("UK Economic Pulse Composite")
st.pyplot(fig, clear_figure=True)


# ----------------------------
# Category scores
# ----------------------------
st.subheader("Three Forces")

cols = st.columns(3)
for i, category in enumerate(["Economic Activity", "Inflation Pressure", "Financial Conditions"]):
    with cols[i]:
        s = category_scores.get(category)
        latest_val = s.dropna().iloc[-1] if s is not None and not s.dropna().empty else np.nan
        st.metric(category, format_latest(latest_val))

        if s is not None and not s.dropna().empty:
            fig_cat = plt.figure(figsize=(4.5, 2.8))
            ax_cat = fig_cat.add_subplot(111)
            ax_cat.plot(s.index, s.values)
            ax_cat.axhline(0, linestyle=":")
            ax_cat.set_title(category)
            ax_cat.set_xlabel("")
            ax_cat.set_ylabel("")
            st.pyplot(fig_cat, clear_figure=True)


# ----------------------------
# Indicator table
# ----------------------------
st.subheader("Indicator Detail")

display = meta[
    ["label", "category", "code", "transform", "latest_level", "latest_signal", "latest_score", "weight"]
].copy()

display = display.rename(
    columns={
        "label": "Indicator",
        "category": "Category",
        "code": "FRED Code",
        "transform": "Transform",
        "latest_level": "Latest Level",
        "latest_signal": "Latest Signal",
        "latest_score": "Score",
        "weight": "Weight",
    }
)

st.dataframe(display, use_container_width=True)


# ----------------------------
# Individual charts
# ----------------------------
st.subheader("Indicator Charts")

selected_key = st.selectbox(
    "Choose an indicator",
    options=list(meta.index),
    format_func=lambda k: f"{SERIES_CONFIG[k].label} ({SERIES_CONFIG[k].code})",
)

spec = SERIES_CONFIG[selected_key]
level = monthly_series[selected_key]
signal = signal_series[selected_key]
score = score_series[selected_key]

left, right = st.columns(2)

with left:
    fig1 = plt.figure(figsize=(6, 3.5))
    ax1 = fig1.add_subplot(111)
    ax1.plot(level.index, level.values)
    ax1.set_title(f"{spec.label} - Level")
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Level")
    st.pyplot(fig1, clear_figure=True)

with right:
    fig2 = plt.figure(figsize=(6, 3.5))
    ax2 = fig2.add_subplot(111)
    ax2.plot(signal.index, signal.values)
    ax2.axhline(0, linestyle=":")
    ax2.set_title(f"{spec.label} - Signal ({spec.transform})")
    ax2.set_xlabel("Date")
    ax2.set_ylabel("Signal")
    st.pyplot(fig2, clear_figure=True)

fig3 = plt.figure(figsize=(12, 3.2))
ax3 = fig3.add_subplot(111)
ax3.plot(score.index, score.values)
ax3.axhline(0, linestyle=":")
ax3.set_title(f"{spec.label} - Score")
ax3.set_xlabel("Date")
ax3.set_ylabel("Score")
st.pyplot(fig3, clear_figure=True)


# ----------------------------
# Methodology
# ----------------------------
with st.expander("Methodology"):
    st.markdown(
        """
        **What this app does**
        - Pulls selected UK macro series from FRED
        - Resamples everything to a monthly backbone
        - Applies a simple transform such as YoY growth or level
        - Converts each transformed series into a rolling robust z-score
        - Flips the sign where higher values are economically bad
        - Aggregates weighted scores into a single composite pulse

        **Interpretation**
        - Positive composite score: broadly supportive macro backdrop
        - Negative composite score: weakening backdrop
        - Thresholds:
          - `>= 0.5` Expansion
          - `<= -0.5` Contraction
          - otherwise Neutral

        **What this is not**
        - It is not a forecasting model
        - It is not a formal recession classifier
        - It is a disciplined composite nowcast-style signal
        """
    )